# Suite BLD — Schema builders as visitors of their meta-schemas

`Builders` holds the meta-schemas (`Builders.Meta`), in which every schema object is a reference object and every
reference between schema objects is a relation, and the builders over them: each a writable `Visitors.OfObject`, with
a member for each property and adjacency of its meta-schema, and VISITOR_PATTERN.md's shorthands. Schema objects and
data are built in a store, `store.moduleName.[...moduleName].schemaName(...)`. See docs/META_SCHEMAS.md, Plan.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Schemas as S
from mbse.Schemas.Framework.Errors import ReadOnlyError
from support import raises

Text, Int, Bool = S.OfScalar.Data(str), S.OfScalar.Data(int), S.OfScalar.Data(bool)


def person(store):
    """crm.Person, who knows other persons since a year, built in `store`."""
    Person = store.Schemas.Object(lambda o: (
        o
        .name("crm.Person")
        .ref(True)
        .properties([
            lambda t: t.name("given_name").type([Text]),
            lambda t: t.name("age").type([Int]),
        ])
    ))
    Knows = store.Schemas.Relation(lambda r: (
        r
        .name("crm.Knows")
        .links([
            lambda t: t.name("knower"),
            lambda t: t.name("known"),
        ])
        .properties([lambda t: t.name("since").type([Int])])
        .uniques("since")
    ))
    store.Schemas.Object(Person, lambda o: o.adjacencies([
        lambda t: t.name("acquaintances").relation([Knows]).me("knower"),
        lambda t: t.relation([Knows]).me("known"),
    ]))
    return Person, Knows

## BLD-01 · The meta-schemas: reference object schemas, and relations between them, each valid

In [ ]:
for name, schema in vars(B.Meta).items():
    if not name.startswith("_"):
        assert schema.validate() == [], (name, schema.validate())
assert B.Meta.Object.ref and list(B.Meta.Object.properties) == ["name", "description", "ref", "pattern", "singleton"]
assert list(B.Meta.Object.adjacencies) == ["parameters", "adjacencies", "properties", "refines"]
assert list(B.Meta.HasProperty.entry.properties) == ["name"]  # a member's name is its entry's
assert B.Meta.HasProperty.uniques == (frozenset({"owner"}), frozenset({"property"}))
assert B.Meta.Typing.uniques == (frozenset({"schema"}),) and B.Meta.AnyOf.uniques == ()
assert all(link.type is None for link in B.Meta.Typing.entry.links.values())  # follows from the adjacencies
assert ("Schemas.Typing", "schema") in [(a.relation.name, a.me) for a in B.Meta.Type.anonymous]
assert "refinement" not in [a.me for a in B.Meta.Type.anonymous]  # only scalars, objects and containers refine
assert B.Meta.Relation.properties["uniques"].type.element.element is not None

## BLD-02 · Building in a store: a callback creates, an object and a callback update, an object alone clones

In [ ]:
store = B.Store()
Person, Knows = person(store)
assert store.registered("crm.Person") is Person and store.registered("crm.Knows") is Knows
assert store.registered("Schemas.Object") is B.Meta.Object  # the store holds the meta-schemas
assert Person.validate() == [] and Knows.validate() == []
bob = store.crm.Person(lambda c: c.given_name("Bob").age(41))
ada = store.crm.Person(lambda c: (
    c
    .given_name("Ada")
    .age(36)
    .acquaintances(lambda adv: adv.insert(lambda env: env.known(bob).since(2020)))
))
assert [str(e.known.given_name) for e in ada.acquaintances] == ["Bob"]
store.crm.Person(ada, lambda c: c.age(37))
assert int(ada.age) == 37
twin = store.crm.Person(ada)
assert twin is not ada and str(twin.given_name) == "Ada" and twin.as_identity() != ada.as_identity()
assert repr(store.crm.Person) == "<crm.Person>"
with raises(TypeError, match="takes a callback, an object and a callback, or an object"):
    store.crm.Person()
with raises(AttributeError, match="no schema registered as 'crm.Nobody'"):
    store.crm.Nobody(lambda c: c)
with raises(AttributeError):
    store._hidden
with raises(AttributeError):
    store.crm._hidden

## BLD-03 · Identities: given on creation, a named schema its name, others a UUID; refused where taken in the store

In [ ]:
store = B.Store()
Person, Knows = person(store)
assert B.identity_of(Person) == "crm.Person" and B.identity_of(B.Meta.Typing) == "Schemas.Typing"
age = Person.properties["age"]
assert len(B.identity_of(age)) == 36 and B.identity_of(age) in store._objects
with raises(ValueError, match="already in this store"):
    store.Schemas.Object(lambda o: o.name("crm.Person"))
store.Schemas.Object(Person, lambda o: o.name("crm.Human"))  # renamed: registered under its new name
assert store.registered("crm.Human") is Person and "crm.Person" not in store.names()
assert store.crm.Human(lambda c: c.given_name("Cy")).as_schema() is Person
store.Schemas.Object(Person, lambda o: o.as_identity("crm.Human.v2"))  # an identity written renames it
assert B.identity_of(Person) == "crm.Human.v2" and store._objects["crm.Human.v2"] is Person
# outside a store
nickname = B.OfProperty.Builder().as_identity("crm.Contact.nickname").type([Text]).create()
assert B.identity_of(nickname) == "crm.Contact.nickname"
builder = B.OfObject.Builder(Person)
assert builder.as_identity() == "crm.Human.v2" and B.OfObject.Builder().as_identity() is None
assert B.identity_of(builder.description("people").update()) == "crm.Human.v2"
with raises(TypeError, match="an identity is non-empty text"):
    builder.as_identity("")
with raises(ReadOnlyError):
    B.OfObject.Builder(Person, False).as_identity("x")
assert B.identity_of(S.OfScalar.Data(int)) is None and B.identity_of("text") is None

## BLD-04 · A schema's members: the peer shorthand, each item linked, or a callback naming the entry and writing its member

In [ ]:
store = B.Store()
spare = store.Schemas.Property(lambda p: p.type([Int]))
Contact = store.Schemas.Object(lambda o: (
    o
    .name("crm.Contact")
    .ref(True)
    .properties([
        lambda t: t.name("given_name").type([Text]),
        lambda t: t.name("spare").property(spare).description("extra"),  # linked, then updated
    ])
))
assert list(Contact.properties) == ["given_name", "spare"] and Contact.properties["spare"] is spare
assert spare.description == "extra" and spare.name == "spare"
with raises(ValueError, match="link the object before writing its members"):
    B.OfObject.Builder().properties([lambda t: t.description("x").property(spare)])
Bare = B.OfObject.Builder().properties([lambda t: t.name("x").property]).create()  # writes nothing: an empty one
assert list(Bare.properties) == ["x"] and Bare.properties["x"].type is None
# the callback form: an entry, its name, and its link
Person = (
    B.OfObject.Builder()
    .name("crm.Person")
    .ref(True)
    .properties(lambda adv: adv.insert(lambda env: env.name("nickname").property(lambda p: p.type([Text]))))
    .create()
)
assert Person.validate() == [] and Person.properties["nickname"].type is Text
with raises(ValueError, match="link 'property' is not set"):
    B.OfObject.Builder().properties(lambda adv: adv.insert(lambda env: env.name("x")))
with raises(TypeError, match="an entry, or a callback taking one, is expected"):
    B.OfObject.Builder().properties(lambda adv: adv.insert(3))
with raises(TypeError, match="takes an iterable of what it links"):
    B.OfAny.Builder().types({"n": Int})
with raises(TypeError, match="link 'property' cannot link 3"):
    B.OfObject.Builder().properties([3])

## BLD-05 · Relations: links by name, their types following from adjacencies; `me` and `uniques` naming links and properties

In [ ]:
store = B.Store()
Person, Knows = person(store)
assert list(Knows.entry.links) == ["knower", "known"] and Knows.uniques == (frozenset({"since"}),)
assert Person.adjacencies["acquaintances"].me == "knower" and [a.me for a in Person.anonymous] == ["known"]
for given, held in (("since", (frozenset({"since"}),)),
                    (["knower", "since"], (frozenset({"knower", "since"}),)),
                    ([["since"], ["knower", "known"]], (frozenset({"since"}), frozenset({"knower", "known"}))),
                    (None, ())):
    assert B.OfRelation.Builder(Knows).uniques(given).clone().uniques == held
builder = B.OfRelation.Builder(Knows).uniques([["since"], ["known", "knower"], ["elsewhere"]])
assert builder.uniques._held() == [["since"], ["knower", "known"], ["elsewhere"]] and builder.uniques.get() is None
assert B.OfRelation.Builder().uniques._held() is None

## BLD-06 · Property shorthands: a value whole, of exactly the property's type; a container's through its visitor

In [ ]:
for wrong in (lambda: B.OfObject.Builder().ref(1), lambda: B.OfObject.Builder().name(3)):
    with raises(TypeError, match="holds"):
        wrong()
call = B.Form.Builder().kind("call").attributes({"name": "knows"}).attributes({"arity": 2}).create()
assert call.attributes == {"arity": 2}  # set whole: nothing merges
with raises(TypeError, match="holds a map"):
    B.Form.Builder().attributes(["x"])
assert B.Form.Builder().attributes({"n": 1.5}).create().attributes == {"n": 1.5}  # an attribute is any scalar
with raises(TypeError, match="holds a list"):
    B.OfRelation.Builder().uniques._matched("since")
with raises(TypeError, match="holds a list"):
    B.OfRelation.Builder().uniques._matched(3)
form = B.Form.Builder().kind("call")
form.attributes(lambda p: p.as_value(lambda v: v.as_container(lambda c: c.at("name").as_scalar("knows"))))
form.attributes.as_value(lambda v: v.as_container(lambda c: c.at("arity").as_scalar(lambda s: s.set(2))))
seen = {}
form.attributes.as_value(lambda v: v.as_container(lambda c: (
    seen.update(length=len(c), schema=c.as_schema()),
    c.as_dict(lambda d: seen.update(keys=list(d))),
    c.as_entries(lambda it: seen.update(values=[e._held() for _, e in it])),
    c.as_ordered(lambda it: seen.update(ordered=True)),
)))
assert seen["length"] == 2 and seen["keys"] == ["name", "arity"] and seen["values"] == ["knows", 2]
assert "ordered" not in seen and seen["schema"] is B.Meta.Form.properties["attributes"].type
form.attributes.as_value(lambda v: v.as_container(lambda c: c.remove("arity")))
form.attributes.as_value(lambda v: v.as_container({"name": "calls"}))
assert form.create().attributes == {"name": "calls"}
form.attributes.as_value(lambda v: v.clear())
assert form.attributes._held() is None
form.kind.as_value(lambda v: (v.as_scalar(lambda s: (s.set("let"), s.get(), s.as_schema())), v.as_schema()))
assert str(form.kind) == "let" and form.kind.get() == "let"
form.kind.as_value(lambda v: v.as_scalar(lambda s: s.clear()))
assert form.kind.get() is None
form.kind.as_scalar("var").as_value(lambda v: v.clear())
assert form.kind.get() is None
with raises(TypeError, match="holds a scalar"):
    form.kind.as_value(lambda v: v.as_object({}))
with raises(TypeError, match="holds a container"):
    form.attributes.as_value(lambda v: v.as_scalar(1))
reasons = []
form.kind.as_object(lambda o: reasons.append(o.get()))
form.attributes.as_value(lambda v: v.as_scalar(lambda s: reasons.append(s.get())))
assert reasons == [None, None] and repr(form.kind) == "<property 'kind'>"

## BLD-07 · Adjacency shorthands: one entry replaced where the relation allows one; `insert` does not replace

In [ ]:
Names = (
    B.OfContainer.Builder()
    .index([Int])
    .element([Text])
    .layout("dense")
    .create()
)
assert Names.validate() == [] and Names.arrangement == "dense"
Keyed = B.OfContainer.Builder(Names).index([Text]).clone()
assert Keyed.index is Text and Names.index is Int
with raises(ValueError, match="remove it before inserting another"):
    B.OfContainer.Builder(Names).index(lambda adv: adv.insert(lambda env: env.index(Text)))
B.OfContainer.Builder(Names).index(lambda adv: adv.insert(lambda env: env.index(Int)))  # held already: elided
with raises(TypeError, match="links an object that exists"):
    B.OfProperty.Builder().type(lambda adv: adv.insert(lambda env: env.schema(lambda s: s)))
with raises(TypeError, match="links an object that exists: link it, not its members"):
    B.OfProperty.Builder().type([lambda t: t.description("x")])
with raises(TypeError, match="link 'schema' cannot link 3"):
    B.OfProperty.Builder().type(lambda adv: adv.insert(lambda env: env.schema(3)))
cleared = B.OfContainer.Builder(Names).index(lambda adv: adv.remove(lambda env: env.index(Int))).clone()
assert cleared.index is None

## BLD-08 · Reading back: members are child visitors; a link reads its referent, read-only

In [ ]:
store = B.Store()
Person, Knows = person(store)
builder = B.OfObject.Builder(Person)
assert builder.name == "crm.Person" and builder.ref.get() is True and builder.name.as_schema().name == "name"
assert [str(env.name) for env in builder.properties] == ["given_name", "age"]
first = next(iter(builder.properties))
assert first.name.as_schema() is B.Meta.HasProperty.entry.properties["name"] and first.as_schema() is B.Meta.HasProperty.entry
assert first.property.as_schema() is B.Meta.Property and first.property.as_identity() == B.identity_of(Person.properties["given_name"])
assert first.property.type.as_schema().relation is B.Meta.Typing and [p.as_schema().name for p in first.property.as_properties] == []
assert repr(first) == "<entry of 'properties'>" and repr(first.property) == "<link 'property'>"
assert repr(builder.properties) == "<adjacency 'properties'>" and repr(builder) == "<builder of an object schema>"
links = []
first.property.as_adjacencies(lambda adv: links.append(adv.as_schema().name))
assert links == ["type"]
assert builder.as_schema() is B.Meta.Object and builder.properties.as_schema() is B.Meta.Object.adjacencies["properties"]
assert [p.as_schema().name for p in builder.as_properties] == ["name", "ref"]
seen = []
builder.as_adjacencies(lambda adv: seen.append(adv.as_schema().name))
assert seen == ["parameters", "adjacencies", "properties", "refines"]
iterated = []
builder.properties.as_iterable(lambda it: iterated.extend(str(e.name) for e in it))
assert iterated == ["given_name", "age"]
with raises(ReadOnlyError, match="read-only"):
    first.property.description("x")
with raises(ReadOnlyError):
    builder.nonesuch("x")
assert first.nonesuch.get() is None and first.property._target() is not None
empty = B.OfObject.Builder().properties(lambda adv: adv.insert(lambda env: (env.property.as_schema(), env.property(lambda p: p))))
assert B.OfAny.Builder().types.as_schema() is B.Meta.Any.adjacencies["types"]
detached = B.OfProperty.Builder()
detached.__class__._meta = "Nonesuch"
assert detached.description.as_schema() is None and detached.type.as_schema() is None
detached.__class__._meta = "Property"
with raises(AttributeError):
    builder._hidden
with raises(AttributeError):
    first._hidden
with raises(AttributeError):
    first.property._hidden
with raises(TypeError, match="is given"):
    B.OfObject.Builder(Text)

## BLD-09 · Editing through an entry: its member updated, or renamed; removing by entry or by template

In [ ]:
store = B.Store()
Person, Knows = person(store)
age = Person.properties["age"]
store.Schemas.Object(Person, lambda o: o.properties(lambda adv: adv.as_iterable(
    lambda it: (next(it), next(it).property(lambda p: p.description("years"))))))
assert Person.properties["age"] is age and age.description == "years"
store.Schemas.Object(Person, lambda o: o.properties(lambda adv: adv.as_iterable(
    lambda it: (next(it), next(it).property(lambda p: p)))))  # writes nothing: nothing changes
store.Schemas.Object(Person, lambda o: o.properties(lambda adv: adv.as_iterable(lambda it: next(it).name("first_name"))))
assert list(Person.properties) == ["first_name", "age"] and Person.validate() == []
store.Schemas.Object(Person, lambda o: o.properties(lambda adv: adv.as_iterable(lambda it: next(it).name.as_scalar("given"))))
assert list(Person.properties) == ["given", "age"]
with raises(TypeError, match="a name is non-empty text"):
    B.OfObject.Builder(Person).properties(lambda adv: adv.as_iterable(lambda it: next(it).name(3)))
with raises(TypeError, match="a name is non-empty text"):
    B.OfObject.Builder(Person).properties([lambda t: t.name("")])
Smaller = (
    B.OfObject.Builder(Person)
    .name("crm.Smaller")
    .properties(lambda adv: adv.remove(lambda env: env.name("age")))
    .clone()
)
assert list(Smaller.properties) == ["given"] and list(Person.properties) == ["given", "age"]
held = next(iter(B.OfObject.Builder(Person).properties))
Fewer = B.OfObject.Builder(Person).properties(lambda adv: adv.remove(held)).clone()
assert list(Fewer.properties) == ["age"]
Typed = B.OfObject.Builder(Person).properties(lambda adv: adv.remove(lambda env: env.property(lambda p: p.type([Int])))).clone()
assert list(Typed.properties) == ["given"]
with raises(TypeError, match="an entry, or a callback taking one, is expected"):
    B.OfObject.Builder(Person).properties(lambda adv: adv.remove(3))
Copied = B.OfObject.Builder().name("demo.Copied").properties(B.OfObject.Builder(Person).properties).create()
assert list(Copied.properties) == ["given", "age"]
Again = B.OfObject.Builder().name("demo.Again").properties(lambda adv: adv.insert(held)).create()
assert list(Again.properties) == ["given"]

## BLD-10 · A clone owns its members, cloned with identities of their own; inherited members stay shared

In [ ]:
store = B.Store()
Person, Knows = person(store)
Twin = store.Schemas.Object(B.OfObject.Builder(Person).as_identity("crm.Twin").name("crm.Twin").clone())
assert Twin is not Person and list(Twin.properties) == ["given_name", "age"]
assert Twin.properties["age"] is not Person.properties["age"] and Twin.properties["age"].type is Int
assert B.identity_of(Twin.properties["age"]) != B.identity_of(Person.properties["age"])
Employee = store.Schemas.Object(lambda o: (
    o
    .name("crm.Employee")
    .refines([Person])
    .properties([lambda t: t.name("badge").type([Text])])
))
assert Employee.ref and list(Employee.properties) == ["given_name", "age", "badge"]
Copy = B.OfObject.Builder(Employee).name("crm.Copy").clone()
assert Copy.properties["age"] is Person.properties["age"] and Copy.properties["badge"] is not Employee.properties["badge"]
assert Copy.adjacencies["acquaintances"] is Person.adjacencies["acquaintances"] and Copy.anonymous == Person.anonymous
eight = B.OfApply.Builder().of([S.OfScalar.Data(int)]).create()
Applied = B.OfApply.Builder(B.OfApply.Builder().of([Person]).arguments(lambda adv: adv).create()).clone()
assert Applied.of is Person

## BLD-11 · Every kind: refinement, parameters, widths as terms, applications, containers, unions, intersections, values of any type, predicates

In [ ]:
store = B.Store()
width = store.Schemas.Parameter(lambda p: p.type([Int]))
Integer = store.Schemas.Scalar(lambda s: (
    s
    .name("Expressions.Domains.Integer")
    .refines([Int])
    .parameters([lambda t: t.name("width").parameter(width)])
    .bits([B.Form.Builder().kind("variable").attributes({"name": "width"}).create()])
))
eight = store.Schemas.Apply(lambda a: (
    a
    .name("Expressions.Domains.int8")
    .of([Integer])
    .arguments(lambda adv: adv.insert(lambda env: env.argument(lambda x: x.parameter([width]).value(8))))
))
Byte = store.Schemas.Scalar(lambda s: (
    s
    .name("demo.Byte")
    .token("int")
    .bits([B.Form.Builder().kind("literal").attributes({"value": 8}).create()])
))
Native = store.Schemas.Scalar(lambda s: s.name("demo.Native").format("python3").token("int").bytes([B.Form.Builder().kind("literal").attributes({"value": 2}).create()]))
assert Integer.token.name == "int" and Integer.bits.kind == "variable" and Byte.bits == 8 and Native.bytes == 2
assert eight.arguments == {"width": 8} and Native.token == S.OfScalar.Token("python3", "int")
literal = next(iter(B.OfScalar.Builder(Byte).bits)).width
assert literal.kind.get() == "literal" and literal.attributes._held() == {"value": 8}
assert B.OfScalar.Builder(Byte).format.get() == "basic" and B.OfScalar.Builder().token.get() is None
assert B.OfScalar.Builder().format.get() is None and B.OfScalar.Builder().format("ccpp").create().token.name == ""
termed = B.OfApply.Builder(eight).arguments(lambda adv: adv.as_iterable(lambda it: next(it).argument(lambda x: (
    x.value.get(), x.term([B.Form.Builder().kind("variable").create()])))))
assert termed.clone().arguments["width"].kind == "variable"
assert next(iter(B.OfApply.Builder(termed.clone()).arguments)).argument.value.get() is None
assert B.OfScalar.Builder(Integer).bits.as_schema().relation is B.Meta.BitWidth
Names = store.Schemas.Container(lambda c: c.name("demo.Names").index([Int]).element([Text]))
Tags = store.Schemas.Container(lambda c: c.name("demo.Tags").refines([Names]))
assert Tags.index is Int and Tags.element is Text and Tags.validate() == []
Short = store.Schemas.Scalar(lambda s: s.name("demo.Short").refines([Integer]))
assert Short.token == Integer.token
Shape = store.Schemas.Union(lambda u: (
    u
    .name("geo.Shape")
    .flat(True)
    .branches([
        lambda t: t.name("count").type([Int]),
        lambda t: t.name("label").type([Text]),
    ])
))
Stamped = store.Schemas.Intersection(lambda i: (
    i
    .name("geo.Stamped")
    .parts([
        lambda t: t.name("area").type([store.Schemas.Object(lambda o: o.name("geo.Area").properties([lambda t: t.name("size").type([Int])]))]),
        lambda t: t.name("when").type([store.Schemas.Object(lambda o: o.name("geo.When").properties([lambda t: t.name("at").type([Int])]))]),
    ])
))
Either = store.Schemas.Any(lambda a: a.types([Int, Text]))
Mutual = store.Schemas.Predicate(lambda p: (
    p
    .name("crm.Mutual")
    .symbols([
        lambda t: t.name("a").type([store.Schemas.Object(lambda o: o.name("crm.Who").ref(True))]),
        lambda t: t.name("b").type([store.registered("crm.Who")]),
    ])
    .requires([B.Form.Builder().kind("call").attributes({"name": "knows"}).operands([B.Form.Builder().kind("variable").create()]).create()])
))
for schema in (Integer, eight, Byte, Shape, Stamped, Either, Mutual):
    assert schema.validate() == [], (schema, schema.validate())
assert [b.name for b in Shape.branches] == ["count", "label"] and list(Mutual.symbols) == ["a", "b"]
assert Either.types == (Int, Text) and Mutual.requires.kind == "call" and len(Mutual.requires.arguments) == 1
assert B.identity_of(Mutual.requires) in store._objects or B.identity_of(Mutual.requires) is not None
with raises(TypeError, match="a form is immutable"):
    B.Form.Builder(Mutual.requires).update()
edited = B.OfPredicate.Builder(Mutual).requires(lambda adv: adv.as_iterable(lambda it: next(it).term(lambda f: f.kind("let")))).clone()
assert edited.requires.kind == "let" and Mutual.requires.kind == "call"
Linked = B.OfAdjacency.Builder().relation(lambda adv: adv.insert(lambda env: env.relation(lambda r: r.name("demo.R").links([lambda t: t.name("a"), lambda t: t.name("b")])))).me("a").create()
assert Linked.relation.name == "demo.R" and Linked.me == "a"

## BLD-12 · Edges: reading through the shorthand's visitor, a scalar's conversions, finalizing in the wrong order

In [ ]:
form = B.Form.Builder().kind("call")
with raises(TypeError, match="holds a scalar"):
    form.kind.as_value(lambda v: v.as_container({}))
with raises(TypeError, match="property 'kind' holds a str, not an int"):
    form.kind.as_value(lambda v: v.as_scalar(lambda s: int(s)))
Int8 = B.OfScalar.Builder().name("demo.I8").refines([Int]).create()
assert B.OfScalar.Builder(Int8).refines([Int]).clone().refines is Int  # held already: elided
seen = []
B.OfObject.Builder().properties([lambda t: (
    seen.append((t.as_schema(), t.name.get())),
    t.name(lambda n: n.as_scalar("x")),
    seen.append((t.name.get(), t.name.as_schema().name)),
)[1]])
assert seen == [(B.Meta.HasProperty.entry, None), ("x", "name")]
with raises(AttributeError):
    B.OfObject.Builder().properties([lambda t: t._hidden])
with raises(AttributeError):
    B.OfObject.Builder().properties([lambda t: t.name._hidden])
with raises(ValueError, match="link 'schema' is not set"):
    B.OfProperty.Builder().type([lambda t: t])
with raises(ValueError, match="create() is only valid without a source instance"):
    B.OfObject.Builder(B.OfObject.Builder().create()).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    B.OfObject.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    B.OfObject.Builder().update()

## BLD-13 · Proxies: a container property's value given whole; the peer shorthand on adjacencies, replacing where one entry is allowed

In [ ]:
store = B.Store()
P = store.Schemas.Object(lambda o: (
    o
    .name("crm.P")
    .ref(True)
    .properties([
        lambda t: t.name("n").type([Text]),
        lambda t: t.name("tags").type([S.OfContainer.Data(index=Int, element=Text)]),
        lambda t: t.name("marks").type([S.OfContainer.Data(index=Text, element=Int)]),
    ])
))
K = store.Schemas.Relation(lambda r: (
    r
    .name("crm.K")
    .links([lambda t: t.name("a"), lambda t: t.name("b")])
    .properties([lambda t: t.name("since").type([Int])])
))
Best = store.Schemas.Relation(lambda r: r.name("crm.Best").links([lambda t: t.name("a"), lambda t: t.name("b")]).uniques("b"))
Three = store.Schemas.Relation(lambda r: r.name("crm.Three").links([lambda t: t.name("a"), lambda t: t.name("b"), lambda t: t.name("c")]))
store.Schemas.Object(P, lambda o: o.adjacencies([
    lambda t: t.name("knows").relation([K]).me("a"),
    lambda t: t.relation([K]).me("b"),
    lambda t: t.name("best").relation([Best]).me("a"),
    lambda t: t.relation([Best]).me("b"),
    lambda t: t.name("trio").relation([Three]).me("a"),
    lambda t: t.relation([Three]).me("b"),
    lambda t: t.relation([Three]).me("c"),
]))
bob = store.crm.P(lambda c: c.n("Bob").tags(["x", "y"]).marks({"math": 3}))
tags, marks = [], []
bob.tags.as_value(lambda v: v.as_container(lambda c: c.as_entries(lambda it: [e.as_scalar(lambda s: tags.append(s.get())) for _, e in it])))
bob.marks.as_value(lambda v: v.as_container(lambda c: c.as_entries(lambda it: [e.as_scalar(lambda s: marks.append((i, s.get()))) for i, e in it])))
assert tags == ["x", "y"] and marks == [("math", 3)]
ada = store.crm.P(lambda c: c.n("Ada").knows([bob, lambda t: t.since(2020).n("Cy")]))
assert [str(e.b.n) for e in ada.knows] == ["Bob", "Cy"] and [e.since.get() for e in ada.knows] == [None, 2020]
store.crm.P(ada, lambda c: c.knows([lambda t: t.b(bob).n("Robert")]))
assert str(bob.n) == "Robert" and len(list(ada.knows)) == 3
store.crm.P(ada, lambda c: c.best([bob]))
store.crm.P(ada, lambda c: c.best([lambda t: t.n("Dee")]))
assert [str(e.b.n) for e in ada.best] == ["Dee"]  # one entry allowed: replaced
store.crm.P(ada, lambda c: c.knows(list(ada.knows)[:1]))  # the entry shorthand: inserted
assert len(list(ada.knows)) == 4
with raises(TypeError, match="have more than two links; give entries"):
    store.crm.P(ada, lambda c: c.trio([bob]))
with raises(TypeError, match="takes an iterable of entries or of what it links"):
    store.crm.P(ada, lambda c: c.knows({"x": bob}))
with raises(ValueError, match="link the object before writing its members"):
    store.crm.P(lambda c: c.knows([lambda t: t.n("Ed").b(bob)]))
with raises(ValueError, match="link 'b' is not set"):
    store.crm.P(lambda c: c.knows([lambda t: t.since(1)]))
seen = []
store.crm.P(lambda c: c.knows([lambda t: (seen.append((t.as_schema() is K.entry, t.since.as_schema().name)), t.n("Fay"))[1]]))
assert seen == [(True, "since")]
with raises(AttributeError):
    store.crm.P(lambda c: c.knows([lambda t: t._hidden]))
with raises(AttributeError):
    store.crm.P(lambda c: c.knows([lambda t: t.n._hidden]))
chained = []
store.crm.P(lambda c: c.knows([lambda t: (chained.append(t.n.as_schema().name), t.n("Gus"))[1]]))
assert chained == ["n"]
with raises(TypeError, match="takes a container of its schema"):
    store.crm.P(lambda c: c.tags("xy"))
hal = store.crm.P(lambda c: c.knows([lambda t: t.b(lambda o: o.n("Hal")).n("Hank")]))  # built at the link, then written
assert [str(e.b.n) for e in hal.knows] == ["Hank"]